## 數據載入與初探

In [1]:
# 載入工具：pandas
import pandas as pd

# 使用相對路徑讀取位於 data 資料夾中的 csv 檔案
# 若文字編碼不是常見的 UTF-8，UnicodeDecodeError 需要特別指定 encoding，如: encoding='ISO-8859-1'
df = pd.read_csv('../data/online_retail.csv')

# 顯示資料的前 5 行，讓我們看看它長什麼樣子
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [2]:
# 顯示資料的基本資訊，例如欄位名稱、資料型態、非空值數量等
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  str    
 1   StockCode    541909 non-null  str    
 2   Description  540455 non-null  str    
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  str    
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 33.1 MB


In [3]:
# 查看數值統計
df.describe()

,Quantity,UnitPrice,CustomerID
count,541909.000000,541909.000000,406829.000000
mean,9.552250,4.611114,15287.690570
std,218.081158,96.759853,1713.600303
min,-80995.000000,-11062.060000,12346.000000
25%,1.000000,1.250000,13953.000000
50%,3.000000,2.080000,15152.000000
75%,10.000000,4.130000,16791.000000
max,80995.000000,38970.000000,18287.000000


## 數據清洗

In [ ]:
# 為了不影響原始數據 df，先複製一份出來進行清洗
# 清洗操作的結果，都存到一個新的變數 df_cleaned 中，以保留原始數據 df 作為對照
df_cleaned = df.copy()
print(f"原始數據維度: {df_cleaned.shape}")

原始數據維度: (541909, 8)


In [5]:
# 清洗任務1: 刪除所有 CustomerID 為空的紀錄
# .dropna() 會刪除任何包含 NaN (Not a Number) 值的行
# subset=['CustomerID'] 表示我們只關心 CustomerID 這一欄是否為空
df_cleaned.dropna(subset=['CustomerID'], inplace=True)
print(f"處理完 CustomerID 缺失值後，數據維度: {df_cleaned.shape}")


處理完 CustomerID 缺失值後，數據維度: (406829, 8)


In [6]:
# 清洗任務2: 刪除所有 Quantity <= 0 的紀錄
# 我們使用條件篩選，只保留 Quantity > 0 的行
df_cleaned = df_cleaned[df_cleaned['Quantity'] > 0]
print(f"處理完 Quantity 異常值後，數據維度: {df_cleaned.shape}")

處理完 Quantity 異常值後，數據維度: (397924, 8)


In [7]:
# 清洗任務3: 刪除所有 UnitPrice <= 0 的紀錄
df_cleaned = df_cleaned[df_cleaned['UnitPrice'] > 0]
print(f"處理完 UnitPrice 異常值後，數據維度: {df_cleaned.shape}")


處理完 UnitPrice 異常值後，數據維度: (397884, 8)


In [8]:
# 清洗任務4: 將 InvoiceDate 從 object 轉為 datetime
df_cleaned['InvoiceDate'] = pd.to_datetime(df_cleaned['InvoiceDate'])

In [10]:
# 清洗任務5: 將 CustomerID 從 float64 轉為 int (因為已經沒有缺失值了)
df_cleaned['CustomerID'] = df_cleaned['CustomerID'].astype(int)

In [11]:
# 清洗完成後，再次呼叫 .info() 來做一次最終的健康檢查
print("數據清洗完成後的最終體檢報告：")
df_cleaned.info()

數據清洗完成後的最終體檢報告：
<class 'pandas.DataFrame'>
Index: 397884 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    397884 non-null  str           
 1   StockCode    397884 non-null  str           
 2   Description  397884 non-null  str           
 3   Quantity     397884 non-null  int64         
 4   InvoiceDate  397884 non-null  datetime64[us]
 5   UnitPrice    397884 non-null  float64       
 6   CustomerID   397884 non-null  int64         
 7   Country      397884 non-null  str           
dtypes: datetime64[us](1), float64(1), int64(2), str(4)
memory usage: 27.3 MB


## 特徵工程

In [12]:
# 我們需要一個 'TotalPrice' 欄位來計算客戶的總消費金額 (Monetary)
df_cleaned['TotalPrice'] = df_cleaned['Quantity'] * df_cleaned['UnitPrice']

In [14]:
# Recency 的計算需要一個 '今天' 的基準點，以確保所有 Recency 都是正數
# 將 '基準日期' 設定為數據集中最後一筆訂單日期的後一天
import datetime as dt
snapshot_date = df_cleaned['InvoiceDate'].max() + dt.timedelta(days=1)
print(f"RFM 分析的基準日期 (Snapshot Date): {snapshot_date}")

RFM 分析的基準日期 (Snapshot Date): 2011-12-10 12:50:00


In [15]:
# 將交易流水帳 (df_cleaned) 聚合成以客戶為單位的 RFM 表
rfm_df = df_cleaned.groupby('CustomerID').agg({
    'InvoiceDate': lambda date: (snapshot_date - date.max()).days, # 計算 Recency：基準日期 - 客戶的最後購買日期
    'InvoiceNo': 'nunique',                                       # 計算 Frequency：計算客戶的獨立訂單數
    'TotalPrice': 'sum'                                           # 計算 Monetary：加總客戶的所有消費金額
})

# 將聚合後的欄位名稱重新命名，使其更直觀
rfm_df.rename(columns={'InvoiceDate': 'Recency',
                       'InvoiceNo': 'Frequency',
                       'TotalPrice': 'Monetary'}, inplace=True)

# 顯示 RFM 表的前 5 行，看看我們的成果
rfm_df.head()


,Recency,Frequency,Monetary
CustomerID,,,
12346,326,1,77183.60
12347,2,7,4310.00
12348,75,4,1797.24
12349,19,1,1757.55
12350,310,1,334.40


## 建模 (K-Means 客戶分群)

In [16]:
#數據標準化
from sklearn.preprocessing import StandardScaler

# 從 rfm_df 中取出我們需要進行分群的 R, F, M 三個欄位
rfm_to_scale = rfm_df[['Recency', 'Frequency', 'Monetary']]

# 建立一個標準化轉換器
scaler = StandardScaler()

# 對數據進行標準化
rfm_scaled = scaler.fit_transform(rfm_to_scale)

# 標準化後的結果是一個 numpy array，我們可以把它轉回 DataFrame 方便查看
import pandas as pd
rfm_scaled_df = pd.DataFrame(rfm_scaled, columns=rfm_to_scale.columns, index=rfm_to_scale.index)

print("數據標準化後的前 5 行：")
rfm_scaled_df.head()


數據標準化後的前 5 行：


,Recency,Frequency,Monetary
CustomerID,,,
12346,2.334574,-0.425097,8.358668
12347,-0.905340,0.354417,0.250966
12348,-0.175360,-0.035340,-0.028596
12349,-0.735345,-0.425097,-0.033012
12350,2.174578,-0.425097,-0.191347


In [17]:
# 建立與訓練 K-Means 模型
from sklearn.cluster import KMeans

# 建立一個 K-Means 模型，我們先嘗試將客戶分為 4 群
# n_clusters=4 是我們告訴演算法要分成 4 群
# random_state=42 是一個隨機種子，確保我們每次執行得到的結果都一樣，方便重現
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)

# 使用標準化後的數據來訓練模型 (找到 4 個群體的中心點)
kmeans.fit(rfm_scaled_df)

# 將分群的結果 (標籤 0, 1, 2, 3) 加回到我們原始的 rfm_df 中
rfm_df['Cluster'] = kmeans.labels_

print("加上分群標籤後的 RFM 表：")
rfm_df.head()


加上分群標籤後的 RFM 表：


,Recency,Frequency,Monetary,Cluster
CustomerID,,,,
12346,326,1,77183.60,3
12347,2,7,4310.00,0
12348,75,4,1797.24,0
12349,19,1,1757.55,0
12350,310,1,334.40,1
